# PROC GVARCLUSによる生産ラインセンサー冗長性の削減

## エグゼクティブサマリー

複数ゾーンから成る製造ラインは、多くが同じ潜在信号を反映する相関の高いセンサー読み取り値を数十チャンネル分ストリーミングしている。本ノートブックは **PROC GVARCLUS**(変数クラスタリング)を用いて13個のプロセスセンサーを4つの排他的クラスタにグループ化し、各クラスタのR二乗値を読み取ってクラスタごとに代表ゲージを1つ選定する — プロセス情報を失うことなくSPC監視対象を削減する。4クラスタのうち3つは物理ゾーンにきれいに対応する(平均R二乗0.92、0.93、0.96)。4つ目は手続きが単独で切り出した単一チャンネルのクラスタであり、見過ごさずに検討する。

## データソース

すべてのデータは `call streaminit(20260531)` と `rand()` によりインラインで生成される — 外部やネットワークからの入力はない。

| データセット | 行数 | 主要変数 | 説明 |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | 3ゾーンの生産ラインから得た合成読み取り値。同一ゾーン内のセンサーは潜在的な駆動因子を共有する(相関が高い)。ゾーン横断のゲージ(温度はゾーン1、圧力はゾーン2、振動はゾーン3に結び付け)を加えて現実的な冗長性を作り出している。DATAステップのループは300サイクルを要求するが、この評価環境はライセンスなしモードで動作し最初の100オブザベーションのみを保持する — クラスタ構造をきれいに復元するには十分な数である。 |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | 入力センサーごとの1行:割り当てられたクラスタとそのクラスタ成分とのR二乗値。`OUTPUT OUT=` ステートメントにより生成される。 |

# PROC GVARCLUSによる生産ラインセンサー冗長性の削減

計装された生産ラインでは、重複する物理現象を計測する数十のセンサーをログすることが一般的である — 1ゾーン内の複数の熱電対、冗長な圧力トランスデューサ、同一シャフトを追跡する振動プローブなど。すべてのチャンネルを管理図や下流モデルに投入すると監視の手間が無駄になり、多重共線性が増大する。

**変数クラスタリング** はこれに直接対応する。`PROC GVARCLUS` は数値変数を *排他的* クラスタに分割し、各クラスタはそのメンバーの第一主成分によって要約される。連動して動くセンサーは同じクラスタに収まり、エンジニアはクラスタごとに代表チャンネルを1つ残すことができる。

本ノートブックでは以下を行う:

1. 意図的に冗長なセンサーを持つ3ゾーンラインの読み取り値を合成する(ループは300サイクルを要求し、ここでは100が保持される)。
2. `MAXCLUSTERS=4` で `PROC GVARCLUS` を用いて13チャンネルをクラスタリングする。
3. クラスタ割当を出力データセットに取得し要約する。
4. R二乗値を解釈し、継続的なSPCのためクラスタごとに1つのゲージを選定する。

## ステップ1 — 合成センサーデータを生成する

隠れた駆動因子(`zone1_a`、`zone2_a`、`zone3_a`)を持つ3つのプロセスゾーンをシミュレートする。残りのチャンネルは各ゾーンの駆動因子のノイズ付き線形関数であるため、ゾーン内では強く相関し、ゾーン間ではほぼ独立である。また、実際のラインで見られる計器間冗長性を模して、入口・出口温度をゾーン1に、2つの圧力トランスデューサをゾーン2に結び付けている。固定シードにより実行は再現可能である。ループは300サイクルを要求するが、ライセンスなしモードではエンジンが最初の100オブザベーションのみを保持し、以下のNOTEがそれを報告する。

In [1]:
データ process_sensors;
    呼出 streaminit(20260531);
    繰返 cycle = 1 から 300;
        /* ゾーン1:潜在駆動因子と2つの冗長プローブ */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* ゾーン2:潜在駆動因子と2つの冗長プローブ */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* ゾーン3:潜在駆動因子と1つの冗長プローブ */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* 既存の駆動因子に結び付けた計器横断チャンネル */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        出力;
    終了;
    削除 cycle;
実行;


NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds


## ステップ2 — センサーをクラスタリングする

13チャンネルすべてに対して `PROC GVARCLUS` を呼び出す。`VAR` ステートメントはクラスタリング対象の数値センサーを列挙する。`MAXCLUSTERS=4` でパーティションを4クラスタに制限する(物理ゾーンごとに概ね1クラスタになると想定し、多少の余裕を持たせている)。`PLOTS=ALL` を伴う `ODS GRAPHICS ON` は変数クラスタのデンドログラムを有効にする。エンジンはそのSVGをインラインで描画するのではなく作業ディレクトリに書き出すため、以下で読み取る構造は印字された **Variable Cluster Assignments** 表とクラスタごとの固有値表に基づく。

`OUTPUT OUT=` ステートメントは、変数からクラスタへの割当を — 各変数の自クラスタに対するR二乗値とともに — 後でプログラムから利用できるデータセットに書き出す。

In [2]:
ODS GRAPHICS ON;

処理 gvarclus データ=process_sensors maxclusters=4 PLOTS=ALL;
    変数 zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    出力 out=clusters;
実行;

ODS GRAPHICS OFF;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## ステップ3 — SUMMARYオプションで再実行する

`SUMMARY` オプションを付けて手続きを再度実行しても同じパーティションが維持される。今回のパスでは `PLOTS=NONE` によりグラフィックスをオフにする。この環境では印字されるレポートはステップ2と同一である — 同じ13行の割当表、同じ4クラスタ、同じ固有値サマリー — ため、このセルは主に `SUMMARY` と `PLOTS=NONE` オプションが構文解析され実行されることを示すものであり、新しい数値を追加するものではない。

In [3]:
処理 gvarclus データ=process_sensors maxclusters=4 summary PLOTS=none;
    変数 zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
実行;


                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## ステップ4 — 出力データセットを確認する

`OUTPUT OUT=` による `clusters` データセットは、センサーごとに割り当てられた `Cluster` と `RSq_Own`(センサーとそのクラスタ成分との相関の二乗)を1行ずつ持つ。`RSq_Own` が高いということは、そのセンサーがクラスタによってよく代表されていることを意味し — クラスタの代表に道を譲って *削除* する理想的な候補である。クラスタ、続いてR二乗で並べ替えることで、各クラスタで最も代表的なセンサーがそのグループの先頭に来るようにする。

In [4]:
処理 並替 データ=clusters out=clusters_ranked;
    基準 Cluster DESCENDING RSq_Own;
実行;

処理 印刷 データ=clusters_ranked 見出;
    変数 Variable Cluster RSq_Own;
    見出 Variable = "センサーチャンネル"
          Cluster  = "クラスタ"
          RSq_Own  = "自クラスタとのR二乗";
実行;


  Obs                    センサーチャンネル          クラスタ                    自クラスタとのR二乗
-----  ---------------------------  ------------  ----------------------------
    1  ZONE1_A                                 1                       0.96867
    2  ZONE1_B                                 1                        0.9189
    3  TEMP_IN                                 1                      0.903394
    4  TEMP_OUT                                1                      0.889519
    5  ZONE2_A                                 2                       0.98364
    6  ZONE2_B                                 2                      0.946708
    7  PRESSURE_A                              2                      0.929356
    8  PRESSURE_B                              2                      0.920915
    9  ZONE2_C                                 2                      0.892405
   10  ZONE3_A                                 3                      0.977237
   11  VIBRATION                               3   


NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## 結果の解釈

このパーティションはラインの物理的構造の大部分を復元しているが、正直に述べるべき注意点が1つある:

- **クラスタ1** は `zone1_a`(R²=0.969)、`zone1_b`(0.919)、および入口・出口温度 `temp_in`(0.903)と `temp_out`(0.890)をまとめている — ゾーン1の潜在信号に駆動される5チャンネルのうち4つである。平均R二乗0.920。
- **クラスタ2** はゾーン2の3チャンネルすべて `zone2_a`(0.984)、`zone2_b`(0.947)、`zone2_c`(0.892)を、ゾーン2の駆動因子に結び付けた2つの圧力トランスデューサ `pressure_a`(0.929)と `pressure_b`(0.921)とともにまとめている。平均R二乗0.935。
- **クラスタ3** は `zone3_a`(0.977)、`zone3_b`(0.949)、`vibration`(0.959)をまとめている。平均R二乗0.962。
- **クラスタ4** は単一チャンネルである:ゾーン1の3番目のプローブである `zone1_c` がR²=1.000で単独に切り出された(単一要素のクラスタは、自明ながら自分自身によって完全に説明される)。ゾーン1の駆動因子を共有しているにもかかわらず、このサンプルサイズでは手続きは `zone1_c` を `zone1_a`/温度グループとは十分に異なると判断し、クラスタ1に統合せず独自のクラスタとした。

複数チャンネルを持つ3つのクラスタはいずれも平均R二乗が **0.90** を上回っており、単一の成分がメンバー間の変動の大部分を説明していることを裏付けている — まさに私たちが集約したい冗長性である。唯一の外れ値 — `zone1_c` がゾーン1の他のメンバーと一緒ではなく単独のクラスタに入ったこと — は、変数クラスタリングがデータ駆動的であることを示す有用な教訓である:サイクル数を増やすか `MAXCLUSTERS` の予算を上げれば境界は変わり得るため、このパーティションはエンジニアリング上の判断の出発点であり、固定的な真実ではない。

**ラインへの対応。** 複数チャンネルを持つ各クラスタ内では、`RSq_Own` が最も高いセンサー(そのクラスタを最も代表するチャンネル)を残し、残りは定常SPCチャート作成から除外または優先度を下げる — 例えばクラスタ2では `zone2_a`、クラスタ3では `zone3_a`。`zone1_c` の単独クラスタは自動的に残すのではなく調査すべき兆候として扱う:個別に監視する前に、それが真に独自の情報を持つのかクラスタリングの人工物にすぎないのかを確認する。この1チャンネルを保留にしても、これにより監視対象の13チャンネルは4チャンネルの監視計画へと集約され、アラームノイズと多重共線性を削減しながらラインの情報内容を維持する。